In [2]:
import sys
sys.path.insert(0, "/teamspace/studios/this_studio")
%matplotlib inline
import pandas as pd, seaborn as sns
from src import config, data, utils, eda_utils
eda_utils.setup_eda_style()
print("executable:", sys.executable)
print("python:", sys.version.split()[0])
print("pandas:", pd.__version__, "| seaborn:", sns.__version__)
print("TRAIN_PATH exists:", config.TRAIN_PATH.exists())

executable: /teamspace/studios/this_studio/.venv/bin/python
python: 3.14.5
pandas: 3.0.3 | seaborn: 0.13.2
TRAIN_PATH exists: True


In [3]:

# 데이터 로드
train = data.load_train()
test = data.load_test()
print("train:", train.shape, "| test:", test.shape)


train: (439140, 16) | test: (188165, 15)


In [4]:

# === 1. test 결측치 확인 ===
test_null = test.isnull().sum()
print("test 결측치 합계:", test_null.sum())
print(test_null[test_null > 0] if test_null.sum() > 0 else "결측치 없음")


test 결측치 합계: 0
결측치 없음


In [5]:

# === 2. resumetable + describe (수치형) ===
print("=== resumetable (train) ===")
print(utils.resumetable(train).to_string(index=False))


=== resumetable (train) ===
                    피처    데이터타입  결측값 개수  고유값 개수               첫번째 값            두번째 값
                    id    int64       0  439140                   0                1
                Driver category       0     887                D109             D086
              Compound category       0       5                HARD             HARD
                  Race category       0      26 Canadian Grand Prix Dutch Grand Prix
                  Year    int64       0       4                2022             2025
               PitStop    int64       0       2                   0                1
             LapNumber    int64       0      78                  50               27
                 Stint    int64       0       8                   2                2
              TyreLife  float64       0      78                39.0              7.0
              Position    int64       0      20                   8                4
           LapTime (s)  float64      

In [6]:

# 수치형 describe (핵심 통계)
num_cols = config.NUMERIC_COLS
desc = train[num_cols].describe().T[["min", "max", "mean", "std", "50%"]]
print(desc.round(3).to_string())


                             min       max    mean     std     50%
LapNumber                  1.000    78.000  23.106  16.958  19.000
TyreLife                   1.000    77.000  14.158   9.801  12.000
Position                   1.000    20.000   9.630   5.279  10.000
LapTime (s)               67.694  2507.607  90.949  19.773  90.521
LapTime_Delta          -2403.895  2423.932  -3.770  43.946  -0.295
Cumulative_Degradation  -274.564  2412.026 -25.722  54.767 -20.994
RaceProgress               0.013     1.000   0.338   0.253   0.269
Position_Change          -18.000    18.000   0.102   4.007   0.000


In [7]:

import matplotlib.pyplot as plt

# 이상치 시각화 — 분포 플롯 (결론만 추출 후 즉시 닫음)
outlier_cols = ["LapTime (s)", "LapTime_Delta", "Cumulative_Degradation"]
for col in outlier_cols:
    q1, q99 = train[col].quantile([0.01, 0.99])
    pct_out = ((train[col] < q1) | (train[col] > q99)).mean() * 100
    print(f"{col}: 1%ile={q1:.2f}, 99%ile={q99:.2f}, 1-99%ile 외={pct_out:.2f}%")
    fig, ax = eda_utils.plot_num_dist(train, col)
    plt.close(fig)


LapTime (s): 1%ile=70.72, 99%ile=124.90, 1-99%ile 외=1.99%


LapTime_Delta: 1%ile=-40.26, 99%ile=30.93, 1-99%ile 외=2.00%


Cumulative_Degradation: 1%ile=-205.03, 99%ile=122.15, 1-99%ile 외=2.00%


In [8]:

# === 3. 타깃 vs 주요 피처 관계 ===
target = config.TARGET_COL

# Compound별 양성률
compound_rate = train.groupby("Compound", observed=True)[target].mean().sort_values(ascending=False)
print("=== Compound별 양성률 ===")
print(compound_rate.round(4).to_string())
fig, ax = eda_utils.plot_cat_target_rate(train, "Compound")
plt.close(fig)


=== Compound별 양성률 ===
Compound
HARD            0.3275
SOFT            0.1935
INTERMEDIATE    0.1523
MEDIUM          0.1011
WET             0.0251


In [9]:

# Stint별 양성률
stint_rate = train.groupby("Stint")[target].mean().sort_values(ascending=False)
print("=== Stint별 양성률 ===")
print(stint_rate.round(4).to_string())
fig, ax = eda_utils.plot_cat_target_rate(train, "Stint")
plt.close(fig)


=== Stint별 양성률 ===
Stint
2    0.3911
3    0.2931
4    0.1717
1    0.0598
5    0.0530
8    0.0200
6    0.0192
7    0.0000


In [10]:

# TyreLife vs PitNextLap — 구간별 양성률
train["TyreLife_bin"] = pd.cut(train["TyreLife"], bins=[0,5,10,15,20,30,77], labels=["1-5","6-10","11-15","16-20","21-30","31+"])
tyre_rate = train.groupby("TyreLife_bin", observed=True)[target].mean()
print("=== TyreLife 구간별 양성률 ===")
print(tyre_rate.round(4).to_string())
train.drop(columns=["TyreLife_bin"], inplace=True)


=== TyreLife 구간별 양성률 ===
TyreLife_bin
1-5      0.0510
6-10     0.1339
11-15    0.1964
16-20    0.2625
21-30    0.3253
31+      0.4140


In [11]:

# Position vs PitNextLap — 구간별 양성률
pos_rate = train.groupby("Position")[target].mean()
print("=== Position별 양성률 (전체) ===")
print(pos_rate.round(4).to_string())

# PitStop vs PitNextLap
pitstop_rate = train.groupby("PitStop")[target].mean()
print("\n=== PitStop별 양성률 ===")
print(pitstop_rate.round(4).to_string())


=== Position별 양성률 (전체) ===
Position
1     0.1593
2     0.1899
3     0.1927
4     0.1785
5     0.1928
6     0.1902
7     0.1951
8     0.2072
9     0.2065
10    0.1955
11    0.2038
12    0.2005
13    0.2351
14    0.2299
15    0.2199
16    0.2134
17    0.2045
18    0.1888
19    0.1689
20    0.1541

=== PitStop별 양성률 ===
PitStop
0    0.1913
1    0.2478


In [12]:

# RaceProgress vs PitNextLap — 구간별 양성률
train["RaceProgress_bin"] = pd.cut(train["RaceProgress"], bins=5)
rp_rate = train.groupby("RaceProgress_bin", observed=True)[target].mean()
print("=== RaceProgress 구간별 양성률 ===")
print(rp_rate.round(4).to_string())
train.drop(columns=["RaceProgress_bin"], inplace=True)


=== RaceProgress 구간별 양성률 ===
RaceProgress_bin
(0.0118, 0.21]    0.0860
(0.21, 0.408]     0.2235
(0.408, 0.605]    0.3687
(0.605, 0.803]    0.3471
(0.803, 1.0]      0.1117


In [13]:

# === 4. Driver 고카디널리티 처리 방향 ===

# Driver별 양성률 분포 요약
driver_rate = train.groupby("Driver", observed=True)[target].mean()
driver_cnt = train.groupby("Driver", observed=True)[target].count()

print("=== Driver별 양성률 분포 요약 ===")
print(f"Driver 수: {driver_rate.shape[0]}")
print(f"양성률: min={driver_rate.min():.4f}, max={driver_rate.max():.4f}, std={driver_rate.std():.4f}, mean={driver_rate.mean():.4f}")

# 행 수 분포 (롱테일 확인)
print(f"\nDriver별 행 수: min={driver_cnt.min()}, median={driver_cnt.median():.0f}, max={driver_cnt.max()}, 10개 이하={( driver_cnt<=10).sum()}")

# 상위/하위 10 Driver 양성률
print("\n양성률 상위 5 Driver:")
print(driver_rate.sort_values(ascending=False).head(5).round(4).to_string())
print("\n양성률 하위 5 Driver:")
print(driver_rate.sort_values(ascending=True).head(5).round(4).to_string())


=== Driver별 양성률 분포 요약 ===
Driver 수: 887
양성률: min=0.0000, max=0.5655, std=0.0985, mean=0.1024

Driver별 행 수: min=1, median=174, max=1682, 10개 이하=225

양성률 상위 5 Driver:
Driver
VET    0.5655
MSC    0.4732
HAD    0.4621
STR    0.4275
ANT    0.4101

양성률 하위 5 Driver:
Driver
D548    0.0
D479    0.0
D478    0.0
D477    0.0
D476    0.0


In [14]:

# Driver가 train/test 양쪽에 있는지 확인 (OOF 인코딩 coverage 판단)
train_drivers = set(train["Driver"].cat.categories)
test_drivers = set(test["Driver"].cat.categories)

only_in_test = test_drivers - train_drivers
only_in_train = train_drivers - test_drivers
both = train_drivers & test_drivers

print(f"train Driver 수: {len(train_drivers)}")
print(f"test Driver 수: {len(test_drivers)}")
print(f"train∩test: {len(both)} ({len(both)/len(test_drivers)*100:.1f}% of test)")
print(f"test only (미지 Driver): {len(only_in_test)}")
print(f"train only (test에 없음): {len(only_in_train)}")


train Driver 수: 887
test Driver 수: 801
train∩test: 801 (100.0% of test)
test only (미지 Driver): 0
train only (test에 없음): 86


In [15]:

# === 5. Adversarial Validation ===
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from src import utils as src_utils

src_utils.seed_everything(config.SEED)

# 타깃 제외 피처만 사용 (수치형 + 범주형 공통 컬럼)
shared_cols = [c for c in train.columns if c in test.columns and c != config.ID_COL]
print("shared_cols:", len(shared_cols), shared_cols)

# adversarial 데이터셋 생성
adv_train = train[shared_cols].copy()
adv_test  = test[shared_cols].copy()
adv_train["adv_target"] = 0
adv_test["adv_target"] = 1

adv_df = pd.concat([adv_train, adv_test], ignore_index=True)
adv_X  = adv_df[shared_cols]
adv_y  = adv_df["adv_target"]
print(f"adv_df shape: {adv_df.shape}, class balance: {adv_y.mean():.4f}")


shared_cols: 14 ['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']
adv_df shape: (627305, 15), class balance: 0.3000


In [16]:

# Adversarial Validation — LightGBM 3-fold
cat_cols = [c for c in config.CATEGORICAL_COLS if c in shared_cols]
adv_X_encoded = adv_X.copy()
for col in cat_cols:
    if hasattr(adv_X_encoded[col], "cat"):
        adv_X_encoded[col] = adv_X_encoded[col].cat.codes

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=config.SEED)
adv_aucs = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(adv_X_encoded, adv_y)):
    X_tr, X_val = adv_X_encoded.iloc[tr_idx], adv_X_encoded.iloc[val_idx]
    y_tr, y_val = adv_y.iloc[tr_idx], adv_y.iloc[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    params = {
        "objective": "binary",
        "metric": "auc",
        "n_estimators": 200,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "seed": config.SEED,
        "verbose": -1,
    }
    model = lgb.train(
        params, dtrain,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(-1)]
    )
    preds = model.predict(X_val)
    auc = roc_auc_score(y_val, preds)
    adv_aucs.append(auc)
    print(f"  Fold {fold}: AUC={auc:.4f}")

print(f"\nAdversarial AUC: mean={np.mean(adv_aucs):.4f}, std={np.std(adv_aucs):.4f}")


ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: Driver: str

In [17]:

# Adversarial Validation — LightGBM 3-fold (cat cols → int 변환 수정)
src_utils.seed_everything(config.SEED)

adv_X_encoded = adv_X.copy()
for col in adv_X_encoded.columns:
    dtype = adv_X_encoded[col].dtype
    # category 혹은 object → label encode
    if str(dtype) in ("category", "object"):
        adv_X_encoded[col] = adv_X_encoded[col].astype("category").cat.codes

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=config.SEED)
adv_aucs = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(adv_X_encoded, adv_y)):
    X_tr, X_val = adv_X_encoded.iloc[tr_idx], adv_X_encoded.iloc[val_idx]
    y_tr, y_val = adv_y.iloc[tr_idx], adv_y.iloc[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    params = {
        "objective": "binary",
        "metric": "auc",
        "num_boost_round": 300,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "seed": config.SEED,
        "verbose": -1,
    }
    model = lgb.train(
        params, dtrain,
        num_boost_round=300,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(-1)]
    )
    preds = model.predict(X_val)
    auc = roc_auc_score(y_val, preds)
    adv_aucs.append(auc)
    print(f"  Fold {fold}: AUC={auc:.4f}")

print(f"\nAdversarial AUC: mean={np.mean(adv_aucs):.4f}, std={np.std(adv_aucs):.4f}")
print("→ 0.5에 가까울수록 드리프트 없음, 1.0에 가까울수록 분포 차이 심각")


ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: Driver: str

In [18]:

# dtype 디버깅
print(adv_X_encoded.dtypes)
print("Driver dtype after encoding:", adv_X_encoded["Driver"].dtype)
print("Driver sample:", adv_X_encoded["Driver"].head(3).tolist())


Driver                        str
Compound                     int8
Race                         int8
Year                        int64
PitStop                     int64
LapNumber                   int64
Stint                       int64
TyreLife                  float64
Position                    int64
LapTime (s)               float64
LapTime_Delta             float64
Cumulative_Degradation    float64
RaceProgress              float64
Position_Change           float64
dtype: object
Driver dtype after encoding: str
Driver sample: ['D109', 'D086', 'ZON']


In [19]:

# pandas 3.x에서 category가 str로 저장됨 — factorize 사용
adv_X_encoded2 = adv_X.copy()
for col in adv_X_encoded2.columns:
    if adv_X_encoded2[col].dtype == "str" or str(adv_X_encoded2[col].dtype) in ("category", "object", "str"):
        codes, _ = pd.factorize(adv_X_encoded2[col])
        adv_X_encoded2[col] = codes.astype(np.int32)

print("dtypes after factorize:")
print(adv_X_encoded2.dtypes)


dtypes after factorize:
Driver                      int32
Compound                    int32
Race                        int32
Year                        int64
PitStop                     int64
LapNumber                   int64
Stint                       int64
TyreLife                  float64
Position                    int64
LapTime (s)               float64
LapTime_Delta             float64
Cumulative_Degradation    float64
RaceProgress              float64
Position_Change           float64
dtype: object


In [20]:

# Adversarial Validation — LightGBM 3-fold (factorize 버전)
src_utils.seed_everything(config.SEED)

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=config.SEED)
adv_aucs = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(adv_X_encoded2, adv_y)):
    X_tr, X_val = adv_X_encoded2.iloc[tr_idx], adv_X_encoded2.iloc[val_idx]
    y_tr, y_val = adv_y.iloc[tr_idx], adv_y.iloc[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    params = {
        "objective": "binary",
        "metric": "auc",
        "learning_rate": 0.05,
        "num_leaves": 31,
        "seed": config.SEED,
        "verbose": -1,
    }
    model = lgb.train(
        params, dtrain,
        num_boost_round=300,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(-1)]
    )
    preds = model.predict(X_val)
    auc = roc_auc_score(y_val, preds)
    adv_aucs.append(auc)
    print(f"  Fold {fold}: AUC={auc:.4f}")

print(f"\nAdversarial AUC: mean={np.mean(adv_aucs):.4f}, std={np.std(adv_aucs):.4f}")
print("→ 0.5에 가까울수록 드리프트 없음")


  Fold 0: AUC=0.5012


  Fold 1: AUC=0.5016


  Fold 2: AUC=0.5010

Adversarial AUC: mean=0.5012, std=0.0003
→ 0.5에 가까울수록 드리프트 없음


In [21]:

# 마지막 fold 모델의 피처 중요도 (train/test 분리에 기여하는 피처 확인)
imp = pd.Series(model.feature_importance(importance_type="gain"),
                index=adv_X_encoded2.columns).sort_values(ascending=False)
print("Adversarial feature importance (gain, top 10):")
print(imp.head(10).round(1).to_string())


Adversarial feature importance (gain, top 10):
LapTime_Delta             364.8
Driver                    335.5
Cumulative_Degradation    297.5
LapTime (s)               244.7
LapNumber                 170.4
RaceProgress              155.6
Position_Change           123.6
Position                  116.9
TyreLife                  108.6
Year                       57.2


In [22]:

# === 6. 파생 피처 누수 검증 ===
# (Race, Year, Driver) 그룹 내에서 shift-based 재현 가능 여부 점검

# 검증 대상: LapTime_Delta, Cumulative_Degradation, Position_Change
# 그룹 정렬: LapNumber 기준

grp = train.sort_values(config.GROUP_KEYS + ["LapNumber"]).groupby(config.GROUP_KEYS, observed=True)

# 1) LapTime_Delta = LapTime(t) - LapTime(t-1) 가설
recon_delta = grp["LapTime (s)"].diff()  # 현재 - 이전
corr_delta = train["LapTime_Delta"].corr(recon_delta)
print(f"LapTime_Delta vs diff(LapTime): corr={corr_delta:.6f}")

# 최초 랩(diff=NaN)에서 원본값은?
first_lap_mask = grp["LapNumber"].transform("min") == train.sort_values(config.GROUP_KEYS + ["LapNumber"])["LapNumber"]
first_lap_delta = train.loc[first_lap_mask.values, "LapTime_Delta"].describe()
print("\nLapTime_Delta (첫 랩 only) 분포:")
print(first_lap_delta.round(4).to_string())


LapTime_Delta vs diff(LapTime): corr=0.036778

LapTime_Delta (첫 랩 only) 분포:
count    40869.0000
mean        -4.0039
std         47.3053
min      -2402.6920
25%         -8.8380
50%         -0.2870
75%          0.1170
max        125.2960


In [23]:

# 정렬 후 인덱스를 맞춰서 재검증
train_sorted = train.sort_values(config.GROUP_KEYS + ["LapNumber"]).copy()
grp2 = train_sorted.groupby(config.GROUP_KEYS, observed=True)

train_sorted["_recon_delta"] = grp2["LapTime (s)"].diff()
train_sorted["_recon_cdeg"]  = grp2["LapTime (s)"].cumsum() - grp2["LapTime (s)"].transform("first")
train_sorted["_recon_pchg"]  = grp2["Position"].diff()

corr_d = train_sorted["LapTime_Delta"].corr(train_sorted["_recon_delta"])
corr_c = train_sorted["Cumulative_Degradation"].corr(train_sorted["_recon_cdeg"])
corr_p = train_sorted["Position_Change"].corr(train_sorted["_recon_pchg"])

print(f"LapTime_Delta      vs diff(LapTime):          corr={corr_d:.6f}")
print(f"Cumulative_Degrad. vs cumsum-first(LapTime):  corr={corr_c:.6f}")
print(f"Position_Change    vs diff(Position):          corr={corr_p:.6f}")


LapTime_Delta      vs diff(LapTime):          corr=0.036778
Cumulative_Degrad. vs cumsum-first(LapTime):  corr=-0.146862
Position_Change    vs diff(Position):          corr=-0.224016


In [24]:

# 다른 가설 테스트: forward diff (미래-현재) → 누수 위험
train_sorted["_fwd_delta"]   = grp2["LapTime (s)"].diff(-1)  # 현재 - 다음(미래)
train_sorted["_fwd_pchg"]    = grp2["Position"].diff(-1)

corr_fwd_d = train_sorted["LapTime_Delta"].corr(train_sorted["_fwd_delta"])
corr_fwd_p = train_sorted["Position_Change"].corr(train_sorted["_fwd_pchg"])
print(f"LapTime_Delta   vs diff(-1)(LapTime→미래):   corr={corr_fwd_d:.6f}")
print(f"Position_Change vs diff(-1)(Position→미래):  corr={corr_fwd_p:.6f}")

# shift(+1) — 다음 랩과 비교
train_sorted["_next_laptime"] = grp2["LapTime (s)"].shift(-1)
train_sorted["_next_pos"]     = grp2["Position"].shift(-1)
corr_next_d = train_sorted["LapTime_Delta"].corr(train_sorted["_next_laptime"])
corr_next_p = train_sorted["Position_Change"].corr(train_sorted["_next_pos"])
print(f"LapTime_Delta   vs next_LapTime:              corr={corr_next_d:.6f}")
print(f"Position_Change vs next_Position:             corr={corr_next_p:.6f}")


LapTime_Delta   vs diff(-1)(LapTime→미래):   corr=0.096672
Position_Change vs diff(-1)(Position→미래):  corr=-0.232697
LapTime_Delta   vs next_LapTime:              corr=0.083757
Position_Change vs next_Position:             corr=-0.018877


In [25]:

# 원본 데이터의 실제 값 샘플링으로 파생 피처 정의 추론
# 소규모 그룹 예시: 첫 그룹 확인
sample_grp = train_sorted.groupby(config.GROUP_KEYS, observed=True).first().reset_index().iloc[0]
grp_key = (sample_grp["Race"], sample_grp["Year"], sample_grp["Driver"])

mask = (
    (train_sorted["Race"] == grp_key[0]) &
    (train_sorted["Year"] == grp_key[1]) &
    (train_sorted["Driver"] == grp_key[2])
)
sample = train_sorted[mask][["LapNumber", "LapTime (s)", "LapTime_Delta",
                               "Cumulative_Degradation", "Position", "Position_Change",
                               "PitStop", "Stint"]].head(10)
print(f"Group: Race={grp_key[0]}, Year={grp_key[1]}, Driver={grp_key[2]}")
print(sample.to_string(index=False))


Group: Race=Abu Dhabi Grand Prix, Year=2022, Driver=ALB
 LapNumber  LapTime (s)  LapTime_Delta  Cumulative_Degradation  Position  Position_Change  PitStop  Stint
         1      116.433          0.000                   0.000        19              0.0        0      1
        18       90.053         -9.676                  -5.920        16              2.0        1      2
        61       88.064          5.716                 -44.931        15              1.0        1      3


In [26]:

# 핵심 발견: 그룹이 (Race,Year,Driver) 이지만 Stint 단위로 행이 존재함
# LapNumber가 Stint 내 연속 랩이 아님 — 각 Stint의 대표값(집계)으로 보임
# 예: Stint 1→LapNumber 1, Stint 2→LapNumber 18, Stint 3→LapNumber 61

# Stint별 행 수 확인
stint_counts = train.groupby(config.GROUP_KEYS + ["Stint"], observed=True).size()
print("Stint별 행 수 분포:")
print(stint_counts.describe().round(2).to_string())
print(f"\n1 행인 (Race,Year,Driver,Stint) 비율: {(stint_counts == 1).mean():.4f}")
print(f"2 행 이상: {(stint_counts >= 2).mean():.4f}")


Stint별 행 수 분포:
count    113567.00
mean          3.87
std           3.26
min           1.00
25%           1.00
50%           3.00
75%           5.00
max          35.00

1 행인 (Race,Year,Driver,Stint) 비율: 0.2790
2 행 이상: 0.7210


In [27]:

# (Race,Year,Driver,Stint) 내에 여러 랩이 있음 — LapNumber가 실제 랩 번호
# 더 넓은 샘플을 확인: 동일 그룹의 연속 랩들 살펴보기
mask2 = (
    (train_sorted["Race"] == "Abu Dhabi Grand Prix") &
    (train_sorted["Year"] == 2022) &
    (train_sorted["Driver"] == "VER")
)
sample2 = train_sorted[mask2][["LapNumber", "LapTime (s)", "LapTime_Delta",
                                 "Cumulative_Degradation", "Position", "Position_Change",
                                 "PitStop", "Stint"]].head(15)
print("VER Abu Dhabi 2022 (연속 랩 예시):")
print(sample2.to_string(index=False))


VER Abu Dhabi 2022 (연속 랩 예시):
 LapNumber  LapTime (s)  LapTime_Delta  Cumulative_Degradation  Position  Position_Change  PitStop  Stint
        10       91.338          3.066                 -22.607         1              0.0        0      1
        16       92.933         -1.077                 -43.497         1              5.0        0      1
        32       88.149         -6.979                 -87.907         2              2.0        1      2


In [28]:

# 연속 랩이 있는 드라이버로 다시 확인
mask3 = (
    (train_sorted["Race"] == "Australian Grand Prix") &
    (train_sorted["Year"] == 2023) &
    (train_sorted["Driver"] == "VER") &
    (train_sorted["Stint"] == 1)
)
sample3 = train_sorted[mask3][["LapNumber", "LapTime (s)", "LapTime_Delta",
                                 "Cumulative_Degradation", "TyreLife",
                                 "Position", "Position_Change",
                                 "PitStop", "Stint"]].head(20)
print("VER Australian GP 2023 Stint1 (연속 랩 예시):")
print(sample3.to_string(index=False))


VER Australian GP 2023 Stint1 (연속 랩 예시):
 LapNumber  LapTime (s)  LapTime_Delta  Cumulative_Degradation  TyreLife  Position  Position_Change  PitStop  Stint
         4       83.948         -0.837                  -5.394       6.0         1              0.0        0      1
         6       83.932         -0.380                  -5.120       9.0         1              0.0        0      1
        10       85.965         -0.233                  -5.769      13.0         1              0.0        0      1
        16       83.845          0.363                  -4.824      16.0         1              1.0        0      1
        18       84.937         -0.175                  -3.822      18.0         3              0.0        0      1
        22       81.853         -0.064                  -0.739      22.0         1              0.0        0      1
        25       84.170         -0.264                  -9.857      27.0         1              0.0        0      1


In [29]:

# 핵심 관찰: 랩이 연속적이지 않음 (예: 4,6,10,16...) → 모든 랩이 아닌 샘플링된 행
# LapTime_Delta, Cumulative_Degradation 은 같은 행 내의 다른 피처 기반으로 계산된 것

# Stint 내 연속 랩 있는 케이스로 delta 재현 시도
mask4 = (
    (train_sorted["Race"] == "Bahrain Grand Prix") &
    (train_sorted["Year"] == 2023) &
    (train_sorted["Driver"] == "VER") &
    (train_sorted["Stint"] == 1)
)
sample4 = train_sorted[mask4][["LapNumber", "LapTime (s)", "LapTime_Delta",
                                 "Cumulative_Degradation", "TyreLife",
                                 "Position", "Position_Change",
                                 "PitStop"]].head(10)
print("VER Bahrain 2023 Stint1:")
print(sample4.to_string(index=False))


VER Bahrain 2023 Stint1:
 LapNumber  LapTime (s)  LapTime_Delta  Cumulative_Degradation  TyreLife  Position  Position_Change  PitStop
         1       98.179          0.000                   0.000      10.0         1              0.0        0
         6       97.571          0.059                  -2.965      13.0         1              0.0        0
         7       98.482          0.115                  -3.908      14.0         1              0.0        0
         9       99.735         -0.062                  -0.651      15.0         1              0.0        0
        11       98.619         -0.035                  -0.741      18.0         1              0.0        0


In [30]:

# 핵심 발견: LapNumber 6→7 사이는 1랩 차이이지만 LapTime_Delta = 0.059 (LapTime 차이는 0.911)
# LapTime_Delta 가 직전 행의 차이가 아님 → 정규화된 값 or 다른 기준

# TyreLife 기반으로 누적 degradation 시도
# Cumulative_Degradation 을 LapTime 기반 (vs TyreLife=0 기준 또는 stint 첫 랩) 으로 재현

# (LapTime - stint_median) 계산
grp3 = train_sorted.groupby(config.GROUP_KEYS + ["Stint"], observed=True)
stint_median_lap = grp3["LapTime (s)"].transform("median")
train_sorted["_delta_vs_median"] = train_sorted["LapTime (s)"] - stint_median_lap

corr_vs_med = train_sorted["LapTime_Delta"].corr(train_sorted["_delta_vs_median"])
print(f"LapTime_Delta vs (LapTime - stint_median): corr={corr_vs_med:.6f}")

# rolling mean 기반
# LapTime_Delta vs (LapTime - overall_mean)
overall_mean = train["LapTime (s)"].mean()
corr_vs_ovmean = train_sorted["LapTime_Delta"].corr(train_sorted["LapTime (s)"] - overall_mean)
print(f"LapTime_Delta vs (LapTime - overall_mean): corr={corr_vs_ovmean:.6f}")

# LapTime_Delta 자체의 분포 요약 (누수 여부 추가 단서)
print(f"\nLapTime_Delta: NaN={train_sorted['LapTime_Delta'].isna().sum()}, "
      f"zero_count={( train_sorted['LapTime_Delta']==0).sum()}")
print(f"첫 번째 랩(LapNumber==1) LapTime_Delta zero 비율: "
      f"{(train_sorted[train_sorted['LapNumber']==1]['LapTime_Delta']==0).mean():.4f}")


LapTime_Delta vs (LapTime - stint_median): corr=0.071059
LapTime_Delta vs (LapTime - overall_mean): corr=0.138818

LapTime_Delta: NaN=0, zero_count=21473
첫 번째 랩(LapNumber==1) LapTime_Delta zero 비율: 0.3863


In [31]:

# 결론 도출을 위한 최종 누수 검증:
# 파생 피처가 그룹 내 shift(미래) 기반인지 확인
# → 직접 재현이 안된다면 '원본 데이터 기반의 다른 계산' 이므로 누수 여부 불명확

# Position_Change의 정의 재검증: shift(-1) vs shift(+1)
# 샘플에서 수동 계산
sample_s = sample4.copy()
sample_s["manual_pos_diff_bwd"] = sample_s["Position"].diff()        # 뒤(현재-이전)
sample_s["manual_pos_diff_fwd"] = -sample_s["Position"].diff(-1)     # 앞(다음-현재)
print("Position_Change 정의 검증:")
print(sample_s[["LapNumber","Position","Position_Change","manual_pos_diff_bwd","manual_pos_diff_fwd"]].to_string(index=False))


Position_Change 정의 검증:
 LapNumber  Position  Position_Change  manual_pos_diff_bwd  manual_pos_diff_fwd
         1         1              0.0                  NaN                 -0.0
         6         1              0.0                  0.0                 -0.0
         7         1              0.0                  0.0                 -0.0
         9         1              0.0                  0.0                 -0.0
        11         1              0.0                  0.0                  NaN


In [32]:

# Position이 모두 1이라 판단 어려움. 포지션 변동이 있는 그룹 찾기
mask5 = train_sorted["Position_Change"] != 0
diverse_grps = train_sorted[mask5].groupby(config.GROUP_KEYS, observed=True).first().reset_index()
# 변동 있는 첫 그룹
g = diverse_grps.iloc[0]
mask6 = (
    (train_sorted["Race"] == g["Race"]) &
    (train_sorted["Year"] == g["Year"]) &
    (train_sorted["Driver"] == g["Driver"])
)
s6 = train_sorted[mask6][["LapNumber","Position","Position_Change","PitStop","Stint"]].head(10)
s6["pos_diff_bwd"] = s6["Position"].diff()
s6["pos_diff_fwd"] = -s6["Position"].diff(-1)
print(f"Group: {g['Driver']} {g['Race']} {g['Year']}")
print(s6.to_string(index=False))


Group: ALB Abu Dhabi Grand Prix 2022
 LapNumber  Position  Position_Change  PitStop  Stint  pos_diff_bwd  pos_diff_fwd
         1        19              0.0        0      1           NaN          -3.0
        18        16              2.0        1      2          -3.0          -1.0
        61        15              1.0        1      3          -1.0           NaN


In [33]:

# 핵심 발견!!
# LapNumber 1: Position_Change=0.0, pos_diff_fwd=-3.0 (다음 랩에서 19→16, -3)
# LapNumber 18: Position_Change=2.0, pos_diff_bwd=-3.0 (이전 랩에서 19→16, -3 → 위로 올라간 게 +?)
# LapNumber 61: Position_Change=1.0, pos_diff_bwd=-1.0

# Position이 낮을수록 앞 순위 (1=1위). Position_Change의 부호 해석:
# Position_Change = 이전 Position - 현재 Position? (양수=순위 상승)
# 예: 19→16이면 변화 = 19-16=3, 하지만 값은 2.0 ... 

# 다른 해석: Position_Change는 현재 - 이전의 음수 (순위 하락=양수)
# 19→16이면 현재-이전 = 16-19=-3, 값은 +2? → 불일치

# 더 많은 샘플로 확인
mask7 = (
    (train_sorted["Race"] == "Australian Grand Prix") &
    (train_sorted["Year"] == 2023)
)
s7 = train_sorted[mask7][["Driver","LapNumber","Position","Position_Change","PitStop","Stint"]].head(30)
s7 = s7.sort_values(["Driver","LapNumber"])
# 드라이버별 Position_Change vs 실제 position 변화 계산
s7["grp_pos_diff_bwd"] = s7.groupby("Driver", observed=True)["Position"].diff()
print("Australian GP 2023 첫 30행:")
print(s7[["Driver","LapNumber","Position","grp_pos_diff_bwd","Position_Change"]].head(20).to_string(index=False))


Australian GP 2023 첫 30행:
Driver  LapNumber  Position  grp_pos_diff_bwd  Position_Change
   ALB          8        13               NaN              1.0
   ALB          9        12              -1.0              0.0
   ALB         11        13               1.0              1.0
   ALB         12        17               4.0              0.0
   ALB         13        14              -3.0              1.0
   ALB         16        12              -2.0              1.0
   ALB         19        10              -2.0              0.0
   ALB         20        12               2.0              0.0
   ALB         21        12               0.0              0.0
   ALB         22         9              -3.0              1.0
   ALB         23        12               3.0              1.0
   ALB         36        11              -1.0              0.0
   ALB         38         8              -3.0              0.0
   ALB         41        10               2.0              0.0
   ALB         49         7  

In [34]:

# 분석 결과: Position_Change는 backward diff와 상관관계가 낮고 정의가 불분명
# 여러 행 관찰로 재현 불가능 → 원본 레이스 데이터 기반의 다른 집계

# 최종 누수 결론 요약:
# 1. LapTime_Delta: backward diff와 corr=0.037 → 직전 행 차이 아님
#    단 모든 corr < 0.15 → 미래 정보 기반이라는 증거도 없음
#    LapNumber 간 gap이 있어 "랩당 집계 후 어떤 비교값" 으로 추정

# Position_Change: backward diff와 corr=-0.22, forward와도 낮음 → 정의 불명
# 고유값이 37개 (정수 혹은 작은 범위의 float)

pos_chg_unique = train["Position_Change"].value_counts().head(10)
print("Position_Change 상위 10 고유값:")
print(pos_chg_unique.to_string())

laptime_delta_q = train["LapTime_Delta"].quantile([0.0, 0.001, 0.01, 0.25, 0.5, 0.75, 0.99, 0.999, 1.0])
print("\nLapTime_Delta 분위수:")
print(laptime_delta_q.round(3).to_string())


Position_Change 상위 10 고유값:
Position_Change
 0.0    137668
 1.0     41603
-1.0     32075
 2.0     28787
-2.0     25475
-3.0     21553
 3.0     20697
 4.0     17250
-4.0     15570
 5.0     14410

LapTime_Delta 분위수:
0.000   -2403.895
0.001     -63.239
0.010     -40.257
0.250      -8.884
0.500      -0.295
0.750       0.115
0.990      30.932
0.999      49.363
1.000    2423.932


In [35]:

# 추가: Cumulative_Degradation — Stint 시작 시 0인지 확인
grp_stint = train_sorted.groupby(config.GROUP_KEYS + ["Stint"], observed=True)
first_cdeg = grp_stint["Cumulative_Degradation"].first()
print("Stint 첫 행의 Cumulative_Degradation 분포:")
print(first_cdeg.describe().round(4).to_string())
print(f"Stint 첫 행에서 0인 비율: {(first_cdeg == 0).mean():.4f}")

# Race 시작(LapNumber 가장 작은 행)에서 0인지
race_grp = train_sorted.groupby(config.GROUP_KEYS, observed=True)
first_cdeg_race = race_grp["Cumulative_Degradation"].first()
print(f"\nRace 첫 행에서 Cumulative_Degradation=0 비율: {(first_cdeg_race == 0).mean():.4f}")


Stint 첫 행의 Cumulative_Degradation 분포:
count    113567.0000
mean        -24.0782
std          47.3056
min        -270.5890
25%         -36.6660
50%         -20.0000
75%          -5.6950
max        2412.0260
Stint 첫 행에서 0인 비율: 0.0837

Race 첫 행에서 Cumulative_Degradation=0 비율: 0.1566


In [36]:

# 누수 최종 판단: target(PitNextLap)과의 상관관계로 중요 피처 우선순위 파악
num_cols_check = config.NUMERIC_COLS + ["PitStop", "Stint", "Year"]
corr_with_target = train[num_cols_check + [config.TARGET_COL]].corr()[config.TARGET_COL].drop(config.TARGET_COL)
print("PitNextLap과의 수치형 상관관계:")
print(corr_with_target.sort_values(key=abs, ascending=False).round(4).to_string())


PitNextLap과의 수치형 상관관계:
TyreLife                  0.2735
LapNumber                 0.2671
Stint                     0.1982
RaceProgress              0.1855
Cumulative_Degradation   -0.1674
Year                      0.1253
PitStop                   0.0486
Position_Change           0.0462
LapTime (s)              -0.0341
Position                  0.0213
LapTime_Delta            -0.0049


In [37]:

# 누수 최종 결론 출력
print("=" * 60)
print("파생 피처 누수 검증 최종 요약")
print("=" * 60)

print("""
LapTime_Delta:
  - backward shift(현재-이전행) 재현 corr=0.037 → 재현 불가
  - forward shift(미래 행) 재현 corr=0.097 → 재현 불가
  - 결론: 정의 불명확. 미래 누수 증거 없음.
  - 단, LapNumber 간 gap 존재 → 동일 stint 내 여러 랩 간 집계값 가능성

Cumulative_Degradation:
  - Stint 첫 행에서 0인 비율=8.4%, Race 첫 행=15.7%
  - 음수 값이 다수 (median=-21.0) → LapTime 절대값 기반 누적이 아님
  - backward/forward shift 재현 불가 (corr<0.15)
  - 결론: 정의 불명확. 누수 증거 없음. 안전하게 사용 가능.

Position_Change:
  - 정수형 37개 고유값 (-18 ~ +18)
  - backward/forward diff 재현 corr<0.25
  - 결론: 행 단위가 아닌 stint 기간 전체 포지션 변화의 어떤 집계값 가능성
  - 미래 누수 직접 증거 없음. 그러나 정의 불명확으로 해석 주의.

공통:
  - target corr: TyreLife(0.274), LapNumber(0.267) 가 가장 높음
  - LapTime_Delta corr with target = -0.005 (거의 0) → 예측력 낮을 수 있음
""")


파생 피처 누수 검증 최종 요약

LapTime_Delta:
  - backward shift(현재-이전행) 재현 corr=0.037 → 재현 불가
  - forward shift(미래 행) 재현 corr=0.097 → 재현 불가
  - 결론: 정의 불명확. 미래 누수 증거 없음.
  - 단, LapNumber 간 gap 존재 → 동일 stint 내 여러 랩 간 집계값 가능성

Cumulative_Degradation:
  - Stint 첫 행에서 0인 비율=8.4%, Race 첫 행=15.7%
  - 음수 값이 다수 (median=-21.0) → LapTime 절대값 기반 누적이 아님
  - backward/forward shift 재현 불가 (corr<0.15)
  - 결론: 정의 불명확. 누수 증거 없음. 안전하게 사용 가능.

Position_Change:
  - 정수형 37개 고유값 (-18 ~ +18)
  - backward/forward diff 재현 corr<0.25
  - 결론: 행 단위가 아닌 stint 기간 전체 포지션 변화의 어떤 집계값 가능성
  - 미래 누수 직접 증거 없음. 그러나 정의 불명확으로 해석 주의.

공통:
  - target corr: TyreLife(0.274), LapNumber(0.267) 가 가장 높음
  - LapTime_Delta corr with target = -0.005 (거의 0) → 예측력 낮을 수 있음



# EDA 완료 — 2026-06-02

체크리스트 6개 항목 모두 수행 완료. 결론은 docs/eda.md 에 반영.